# 推理 Infra 学习实录 06：从推理服务到魔搭创空间

> 归档状态（GitHub 发布副本）：部署草案，尚未完成目标创空间验证，不代表已上线。 所有 ModelScope 发布 URL 尚未知，不表示已发布。
> [配套文章](article.md) · [上一篇](../05/article.md) · [下一篇](../07/article.md)。Notebook 输出与执行计数已清空。
> 版本边界：mini-SGLang 固定版本 `9a91cfafe754aa85daee49998176275667eb58f2`。
> 路径说明：`/mnt/workspace` 是 ModelScope 平台缓存示例，请按实际环境调整，模型目录以 `snapshot_download()` 返回值为准。

这份 Notebook 将 mini-SGLang 的 OpenAI-compatible API 组织为魔搭创空间应用。内容覆盖进程边界、工作负载、缓存目录、服务检查和部署草案校验。

显存比例、并发和上下文长度是部署初值。目标 GPU、CUDA 兼容性、峰值显存和启动时间需要在实际创空间中验证。

## 1. 运行环境检查

部署记录需要包含实际 GPU、可见显存、PyTorch 和 CUDA 版本。

In [ ]:
!nvidia-smi || true

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("Visible VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except ImportError:
    print("PyTorch is not installed in the current kernel.")

## 2. 应用结构

创空间对外只暴露 Gradio 的 `7860` 端口。mini-SGLang 监听容器回环地址 `127.0.0.1:1919`。

`1919` 仅内部可达不等于鉴权；Gradio 页面与事件/API 仍需访问控制、限流及私有可见性。`show_api=False` 只隐藏 API 文档，不会禁用 API；`api_key="dummy"` 也不是有效鉴权。

In [ ]:
architecture = """
Browser
  -> Gradio 0.0.0.0:7860
     -> OpenAI-compatible API 127.0.0.1:1919/v1
        -> mini-SGLang
           -> Qwen/Qwen3-0.6B
"""
print(architecture)

## 3. 工作负载与资源护栏

初始参数面向单用户交互 Demo，不代表模型能力上限或最优吞吐配置。

In [ ]:
%pip install -q pandas

import pandas as pd

workload = pd.DataFrame([
    ("model", "Qwen/Qwen3-0.6B", "ModelScope public model"),
    ("max context", 4096, "Demo boundary"),
    ("recommended output", 256, "tokens"),
    ("Gradio concurrency", 1, "single interactive task"),
    ("queue size", 8, "reject unbounded queueing"),
    ("max running requests", 8, "mini-SGLang scheduler limit"),
    ("memory ratio", 0.70, "initial value; measure on target GPU"),
], columns=["item", "initial value", "note"])
workload

## 4. 持久化缓存目录

模型和 kernel 缓存写入 `/mnt/workspace`，用于验证缓存重启路径。

In [ ]:
from pathlib import Path

workspace = Path("/mnt/workspace") if Path("/mnt/workspace").exists() else Path("./.studio-workspace")
cache_dirs = {
    "modelscope": workspace / ".cache/modelscope",
    "flashinfer": workspace / ".cache/flashinfer",
    "tvm_ffi": workspace / ".cache/tvm-ffi",
}
for path in cache_dirs.values():
    path.mkdir(parents=True, exist_ok=True)
cache_dirs

## 5. 生成后端启动命令

后端只监听回环地址。创空间外部流量由 Gradio 接收。

In [ ]:
server_command = [
    "python", "-m", "minisgl",
    "--model", "Qwen/Qwen3-0.6B",
    "--model-source", "modelscope",
    "--host", "127.0.0.1",
    "--port", "1919",
    "--cache-type", "radix",
    "--memory-ratio", "0.70",
    "--max-running-requests", "8",
    "--max-seq-len-override", "4096",
    "--cuda-graph-max-bs", "8",
    "--max-prefill-length", "2048",
    "--page-size", "16",
]
print(" \\\n  ".join(server_command))

## 6. 检查 OpenAI-compatible API

以下单元不会自动启动模型服务。环境变量 `RUN_MINISGL_CLIENT=1` 用于显式开启连接测试。

In [ ]:
%pip install -q "openai>=1.50,<2" pyyaml jsonschema

In [ ]:
import os

if os.getenv("RUN_MINISGL_CLIENT") == "1":
    from openai import OpenAI

    client = OpenAI(base_url="http://127.0.0.1:1919/v1", api_key="dummy")
    print(client.models.list())
else:
    print("Skipped. Start mini-SGLang and set RUN_MINISGL_CLIENT=1 to enable the check.")

## 7. 流式对话请求

Gradio 使用同一段 OpenAI-compatible 调用，不直接导入 mini-SGLang。

In [ ]:
if os.getenv("RUN_MINISGL_CLIENT") == "1":
    response = client.chat.completions.create(
        model="Qwen/Qwen3-0.6B",
        messages=[{"role": "user", "content": "用三句话解释 KV Cache。"}],
        temperature=0.6,
        max_tokens=128,
        stream=True,
    )
    for chunk in response:
        text = chunk.choices[0].delta.content
        if text:
            print(text, end="", flush=True)
else:
    print("Skipped streaming request.")

## 8. 校验创空间草案

代码检查覆盖 Python 语法、Shell 语法、必需文件和评估记录结构。

In [ ]:
import ast
import subprocess

candidates = [Path.cwd(), *Path.cwd().parents]
demo_dir = next(
    (candidate / relative
     for candidate in candidates
     for relative in ("studio-demo", "nano-vllm/lessons/06/studio-demo")
     if (candidate / relative / "studio-assessment.yaml").is_file()),
    None,
)
if demo_dir is None:
    raise FileNotFoundError("请从本合集目录或第 06 篇目录运行；未找到 studio-demo 草案。")

required_files = [
    "Dockerfile", "app.py", "entrypoint.sh",
    "requirements.txt", "README.md", "studio-assessment.yaml",
]
for name in required_files:
    path = demo_dir / name
    assert path.is_file(), f"missing: {path}"

# 只做静态检查，不向 studio-demo 写入 __pycache__。
ast.parse((demo_dir / "app.py").read_text(), filename=str(demo_dir / "app.py"))
subprocess.run(["bash", "-n", str(demo_dir / "entrypoint.sh")], check=True)
print("syntax checks passed")

In [ ]:
import yaml

assessment = yaml.safe_load((demo_dir / "studio-assessment.yaml").read_text())
print("assessment:", assessment["assessment"]["id"])
print("recommendation:", assessment["decision"]["recommendation"])
print("open unknowns:")
for item in assessment["unknowns"]:
    if item["status"] == "open":
        print("-", item["question"])

## 9. 部署验收记录

私有创空间完成以下记录后，再确定公开参数：

```text
actual GPU / visible VRAM
CUDA driver / runtime / PyTorch
cold-start time
cached restart time
peak VRAM at recommended workload
single-turn / multi-turn / streaming checks
remaining runtime network calls
```

## 10. 结论

```text
ModelScope Studio Docker
├── Gradio 0.0.0.0:7860
├── mini-SGLang 127.0.0.1:1919
├── Qwen/Qwen3-0.6B
└── /mnt/workspace persistent cache
```

当前目录提供部署草案，评估结论为 `needs_information`。目标环境的运行时探测、峰值显存和缓存重启验证完成后，才能形成已验证的创空间配置。